# DRLB smooth train/test run

This notebook trains the smooth DRLB variant on the original BAT train split and evaluates it on the original BAT test split.

It mirrors the usual baseline-comparison flow:
1. Load the original `train` and `test` data.
2. Train `DRLBBidder` with the smooth DQN flag.
3. Save model, diagnostics, and metrics into a dedicated experiment folder.
4. Report the same test metrics used in baseline comparisons: `SCR`, `CPC_REL`, `RMSE`, and `quickspend`.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

notebook_dir = Path().resolve()
example_notebooks_dir = notebook_dir.parent.parent
bat_autobidding_dir = example_notebooks_dir.parent

if str(example_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(example_notebooks_dir))
if str(bat_autobidding_dir) not in sys.path:
    sys.path.insert(0, str(bat_autobidding_dir))

from experiments.exp_configs import RND42TrainTestHybridSmoothConfig
from simulator.model.drlb_bidder import DRLBBidder
from simulator.validation.check_results import autobidder_check

pd.set_option("display.max_columns", 200)


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = RND42TrainTestHybridSmoothConfig()
config.ensure_artifact_dirs()
config.config_dir.mkdir(parents=True, exist_ok=True)

OBJECTIVE = "clicks"
SMOOTH_EXP_TYPE = "improved_hybrid_drlb_smooth_eval"
MAX_TRAIN_STEPS = None  # Set e.g. 24 * 14 for a faster smoke run.

BASE_DRLB_PARAMS = {
    "max_bid": 100.0,
    "T": 72,
    "lambda_min": 1e-6,
    "lambda_max": 10.0,
    "bids_per_timestep": 1,
    "dqn_soft_update_tau": 0.01,
    "dqn_loss_type": "smooth_l1",
    "dqn_grad_clip_norm": 5.0,
    "dqn_reward_clip_value": 10.0,
}

MODEL_PARAMS = {
    "dqn_gamma": 1.0,
    "dqn_lr": 1e-4,
    "dqn_target_update_interval": 100,
    "reward_net_lr": 1e-3,
}

RUN_LABEL = "train_test_manual"
VERBOSE = False

print("Train campaigns:", config.data_config["train"]["campaigns_path"])
print("Train stats:", config.data_config["train"]["stats_path"])
print("Test campaigns:", config.data_config["test"]["campaigns_path"])
print("Test stats:", config.data_config["test"]["stats_path"])
print("Outputs dir:", config.outputs_dir)


Train campaigns: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_filtered_train_final.csv
Train stats: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_train_final.csv
Test campaigns: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_filtered_test_final.csv
Test stats: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_filtered_test_final.csv
Outputs dir: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_train_test_drlb_dqn_smooth/outputs


## Data overview

This cell loads the original BAT `train` and `test` splits and shows the basic scale of each split before training.


In [3]:
train_stats_df = pd.read_csv(config.data_config["train"]["stats_path"])
test_stats_df = pd.read_csv(config.data_config["test"]["stats_path"])
train_campaigns_df = pd.read_csv(config.data_config["train"]["campaigns_path"])
test_campaigns_df = pd.read_csv(config.data_config["test"]["campaigns_path"])

summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "campaigns": int(train_campaigns_df["campaign_id"].nunique()),
            "stats_rows": int(len(train_stats_df)),
            "unique_periods": int(train_stats_df["period"].nunique()),
        },
        {
            "split": "test",
            "campaigns": int(test_campaigns_df["campaign_id"].nunique()),
            "stats_rows": int(len(test_stats_df)),
            "unique_periods": int(test_stats_df["period"].nunique()),
        },
    ]
)

display(summary_df)


,split,campaigns,stats_rows,unique_periods
0,train,1284,1533171,351
1,test,1285,1503083,267


In [4]:
def score_to_dict(score, skipped_campaigns, time_inference_sec, time_overall_sec):
    return {
        "cpc_relative": float(score[0]),
        "rmse": float(score[1]),
        "clicks_sum": float(score[2]),
        "quickspend": float(score[3]),
        "skipped_campaigns": int(skipped_campaigns),
        "time_inference_sec": float(time_inference_sec),
        "time_overall_sec": float(time_overall_sec),
    }


def summarize_diagnostics(diagnostics_df):
    if diagnostics_df.empty:
        return {
            "train_steps": 0,
            "last_dqn_loss": None,
            "last_reward_net_loss": None,
            "dqn_loss_mean": None,
            "dqn_loss_p95": None,
            "reward_net_loss_mean": None,
            "reward_net_loss_p95": None,
            "reward_signal_mean": None,
            "lambda_final": None,
        }

    dqn_loss_nonzero = diagnostics_df.loc[diagnostics_df["dqn_loss"] > 0, "dqn_loss"]
    reward_net_nonzero = diagnostics_df.loc[diagnostics_df["reward_net_loss"] > 0, "reward_net_loss"]
    return {
        "train_steps": int(len(diagnostics_df)),
        "last_dqn_loss": float(diagnostics_df["dqn_loss"].iloc[-1]),
        "last_reward_net_loss": float(diagnostics_df["reward_net_loss"].iloc[-1]),
        "dqn_loss_mean": float(dqn_loss_nonzero.mean()) if not dqn_loss_nonzero.empty else None,
        "dqn_loss_p95": float(dqn_loss_nonzero.quantile(0.95)) if not dqn_loss_nonzero.empty else None,
        "reward_net_loss_mean": float(reward_net_nonzero.mean()) if not reward_net_nonzero.empty else None,
        "reward_net_loss_p95": float(reward_net_nonzero.quantile(0.95)) if not reward_net_nonzero.empty else None,
        "reward_signal_mean": float(diagnostics_df["reward_signal"].mean()),
        "lambda_final": float(diagnostics_df["lambda"].iloc[-1]),
    }


def build_bidder_params(base_params, model_params, verbose=False):
    return {
        **base_params,
        **model_params,
        "model_path": None,
        "exp_type": SMOOTH_EXP_TYPE,
        "objective": OBJECTIVE,
        "eval_mode": True,
        "verbose": verbose,
        "use_tqdm": verbose,
        "debug_logs": False,
        "fit_log_every": 500,
        "inference_log_every": 24,
    }


def run_train_test(label, model_params, max_train_steps=None, verbose=False):
    bidder_params = build_bidder_params(BASE_DRLB_PARAMS, model_params, verbose=verbose)
    bidder = DRLBBidder(bidder_params)
    bidder.fit(
        train_stats_df,
        campaigns_df=train_campaigns_df,
        max_steps=max_train_steps,
        objective=OBJECTIVE,
    )

    diagnostics = bidder.get_training_diagnostics().copy()
    diagnostics_path = config.outputs_dir / f"{label}_training_diagnostics.csv"
    diagnostics.to_csv(diagnostics_path, index=False)

    model_path = config.best_models_dir / f"{label}.pt"
    bidder.save_model(str(model_path))

    eval_params = {
        **bidder_params,
        "input_campaigns": config.data_config["test"]["campaigns_path"],
        "input_stats": config.data_config["test"]["stats_path"],
        "model_path": str(model_path),
        "eval_mode": True,
    }
    result = autobidder_check(
        bidder=DRLBBidder,
        params=eval_params,
        auction_mode=config.auction_mode,
        verbose=verbose,
        log_every_campaigns=100,
        use_tqdm=verbose,
    )

    metrics = score_to_dict(
        score=result["score"],
        skipped_campaigns=result["skipped_campaigns"],
        time_inference_sec=result["time_inference_sec"],
        time_overall_sec=result["time_overall_sec"],
    )
    metrics.update({"label": label, **model_params})
    metrics.update(summarize_diagnostics(diagnostics))

    metrics_path = config.outputs_dir / f"{label}_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))

    return {
        "label": label,
        "metrics": metrics,
        "diagnostics": diagnostics,
        "model_path": model_path,
        "diagnostics_path": diagnostics_path,
        "metrics_path": metrics_path,
    }


## Train on original train and score on original test

Edit `MODEL_PARAMS`, `RUN_LABEL`, or `MAX_TRAIN_STEPS` above if needed, then run this cell.


In [5]:
run = run_train_test(
    label=RUN_LABEL,
    model_params=MODEL_PARAMS,
    max_train_steps=MAX_TRAIN_STEPS,
    verbose=VERBOSE,
)

metrics_df = pd.DataFrame([run["metrics"]])
display(metrics_df)
display(run["diagnostics"].tail())

print("Model path:", run["model_path"])
print("Diagnostics path:", run["diagnostics_path"])
print("Metrics path:", run["metrics_path"])
print("Score tuple (CPC_REL, RMSE, SCR, quickspend):", [
    run["metrics"]["cpc_relative"],
    run["metrics"]["rmse"],
    run["metrics"]["clicks_sum"],
    run["metrics"]["quickspend"],
])


,cpc_relative,rmse,clicks_sum,quickspend,skipped_campaigns,time_inference_sec,time_overall_sec,label,dqn_gamma,dqn_lr,dqn_target_update_interval,reward_net_lr,train_steps,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,dqn_loss_p95,reward_net_loss_mean,reward_net_loss_p95,reward_signal_mean,lambda_final
0,6404.741892,1.237307,8573.391295,0.0,0,74.080613,95.411938,train_test_manual,1.0,0.0001,100,0.001,60718,0.995476,71.576393,222.483058,1058.622833,525512.359563,304.655487,4.006017,0.000011


,global_t,rem_budget,lambda,eps,dqn_action,dqn_loss,reward_signal,reward_net_loss
60713,60714,0,0.000011,0.05,5,0.520396,5.476171,23.109634
60714,60715,0,0.000010,0.05,0,0.928846,5.476107,266.728271
60715,60716,0,0.000011,0.05,6,0.277294,5.476114,60.020885
60716,60717,0,0.000012,0.05,5,0.071077,5.476107,57.404568
60717,60718,0,0.000011,0.05,1,0.995476,5.475994,71.576393


Model path: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_train_test_drlb_dqn_smooth/best_models/train_test_manual.pt
Diagnostics path: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_train_test_drlb_dqn_smooth/outputs/train_test_manual_training_diagnostics.csv
Metrics path: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/exp_train_test_drlb_dqn_smooth/outputs/train_test_manual_metrics.json
Score tuple (CPC_REL, RMSE, SCR, quickspend): [6404.74189225005, 1.2373070418156094, 8573.391295076146, 0.0]
